<a href="https://colab.research.google.com/github/bei931016/MachineLearning/blob/main/0709_Colab_LINE_Bot_with_GEMINI_Tooluse%20copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [29]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [30]:
import os
from pyngrok import ngrok

In [31]:
ngrok.kill()

In [32]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://extinct-perkiness-shrapnel.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://extinct-perkiness-shrapnel.ngrok-free.dev


True

In [33]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [34]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [35]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology, MUST）是一所位於臺灣新竹縣新豐鄉的私立科技大學。學校佔地逾三十公頃，地處新竹科學園區與新竹工業區的產業資源地帶，交通便利，被定位為一所「產業大學」，著重於培育實務人才並強化產學連結。

**歷史沿革：**
明新科技大學的創校可追溯至1966年，當時名為「明新工業專科學校」，初期設有機械、土木、工業管理等工業類專科課程。
1993年，更名為「明新工商專科學校」，並增設國際貿易、企業管理等商業課程。
1997年，經教育部核准改制為「明新技術學院」，開始提供學士學位課程，並增設旅館事業管理、幼兒保育等服務產業相關學程。
2002年9月，教育部核准其升格為「明新科技大學」，並設有工程學院、管理學院、服務事業學院三個學院（現已發展為半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院共六個學院），同時增設碩士學位課程，邁向綜合型大學發展。 2012年，增設學士後行銷與流通管理學士學位學程、學士後旅館管理實務學士學位學程。 2022年，明新科大獲教育部核准通過「半導體科技博士學位學程」，為該校成立五十六年來第一個博士班。

**辦學特色與現況：**
明新科技大學秉持「堅毅、求新、創造」的校訓，以中華傳統人文精神為基礎，致力於培養具備國際視野、區域發展特色，且兼具教學與創新應用能力的科技大學。 學校提供超過27個學術領域，涵蓋工程、商業、服務管理和社會科學。

目前，明新科技大學擁有約12,000名學生（日間部與夜間部合計超過12,000名），以及約80,000名校友遍布全球。 學校被譽為臺灣排名前列的私立科技大學之一，以其卓越的辦學績效和受企業青睞的畢業生而聞名。 校園內教學設施現代化，包括高速無線網路、高科技實驗室、圖書館、電腦中心、專業教室、實習餐廳以及各類現代化運動設施等。 學校也積極推動產學合作，透過契合式人才課程，提升畢業生的就業競爭力。


In [36]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[]}
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:36:27] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[]}


INFO:__main__:Request body: {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617022570298343578","quoteToken":"hbZH5kVS8WogwmvuNa0npHC-mjDwBNV1lEocWFjxOUQnp4BaRodw8opvCpv3MOZZvLxQeTH3JBOcgT3NjvkeKJWmZN8YtlK8gyITbvp-wiz01OgZehm33O0UAC-wx5eVqJ5mwJoZ2hlWWW0UIhI-Eg","markAsReadToken":"61FPTQfYMSGhZhJXydQFIW0b450QczK1_UdBLd3V3VIzA0xAV5ZLtpApX2F6uTz5QKg3WSCS_Zacsro5eIPVg3rCrqEKD_ZYe1sHoN_CiQFewQiT-bkhRzUDC5_AvNo-RLUbucT8zHqTqhgJFdukBMb98n8WBJJLfu09aM7qCgK7QFBtKDldU5DRGJsQvGHE3jhG-zxKHhoWflY9VNvXmw","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTA5MW9H1VYKADHEEG5Y4J5H","deliveryContext":{"isRedelivery":false},"timestamp":1780605415222,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"cd5504edae2c4ae9a4e5c4c87277f36d","mode":"active"}]}


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617022570298343578","quoteToken":"hbZH5kVS8WogwmvuNa0npHC-mjDwBNV1lEocWFjxOUQnp4BaRodw8opvCpv3MOZZvLxQeTH3JBOcgT3NjvkeKJWmZN8YtlK8gyITbvp-wiz01OgZehm33O0UAC-wx5eVqJ5mwJoZ2hlWWW0UIhI-Eg","markAsReadToken":"61FPTQfYMSGhZhJXydQFIW0b450QczK1_UdBLd3V3VIzA0xAV5ZLtpApX2F6uTz5QKg3WSCS_Zacsro5eIPVg3rCrqEKD_ZYe1sHoN_CiQFewQiT-bkhRzUDC5_AvNo-RLUbucT8zHqTqhgJFdukBMb98n8WBJJLfu09aM7qCgK7QFBtKDldU5DRGJsQvGHE3jhG-zxKHhoWflY9VNvXmw","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTA5MW9H1VYKADHEEG5Y4J5H","deliveryContext":{"isRedelivery":false},"timestamp":1780605415222,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"cd5504edae2c4ae9a4e5c4c87277f36d","mode":"active"}]}


...

INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:41:55] "POST / HTTP/1.1" 200 -


檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/f69y8gyb5zca


INFO:__main__:Request body: {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617023087741239349","quoteToken":"S_QEapaXXZ4EaRFN7UMVPrqdSnf6SndSkz7lbBI0efZhjfsi0Ajp2U01YL2lIWAbxZ7CwsXwq8N3di-V3z_uZD_OEUwtpp4j0K5p8B8dpDgktge7ID3wiVZF74PdsiKcd60TEr6VJEW2VVgJn5iBEQ","markAsReadToken":"5mmiORm4TMEM4jJkBCtcBHNyZK3IyqlbZdNjldLFN7IcUOvJ7B9Q3rC4onDZfCtinQ3ir59MmylvmjW0urvp36uBXTMdDbl_nqRlSGaouKmMn1QMOts8vfR962hq_v5IHdXvm79YOnNvau0gxJuOYg8NUWSH9WJcyotDKhy2n6jxUKUs5XSPNmk-iRCoLfTNWXi8W4sxONMNsCvP1ynxhg","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTA5Y9FGS57RWFT6AYH9BGG2","deliveryContext":{"isRedelivery":false},"timestamp":1780605723637,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"71a999bb577e4517911e2a9a0042f6b1","mode":"active"}]}


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617023087741239349","quoteToken":"S_QEapaXXZ4EaRFN7UMVPrqdSnf6SndSkz7lbBI0efZhjfsi0Ajp2U01YL2lIWAbxZ7CwsXwq8N3di-V3z_uZD_OEUwtpp4j0K5p8B8dpDgktge7ID3wiVZF74PdsiKcd60TEr6VJEW2VVgJn5iBEQ","markAsReadToken":"5mmiORm4TMEM4jJkBCtcBHNyZK3IyqlbZdNjldLFN7IcUOvJ7B9Q3rC4onDZfCtinQ3ir59MmylvmjW0urvp36uBXTMdDbl_nqRlSGaouKmMn1QMOts8vfR962hq_v5IHdXvm79YOnNvau0gxJuOYg8NUWSH9WJcyotDKhy2n6jxUKUs5XSPNmk-iRCoLfTNWXi8W4sxONMNsCvP1ynxhg","text":"AI 校長愛吃什麼"},"webhookEventId":"01KTA5Y9FGS57RWFT6AYH9BGG2","deliveryContext":{"isRedelivery":false},"timestamp":1780605723637,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"71a999bb577e4517911e2a9a0042f6b1","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:42:07] "POST / HTTP/1.1" 200 -
